In [0]:
%pip install openpyxl
dbutils.library.restartPython()

In [0]:
import pandas as pd

FILE_PATH = "/Workspace/Users/lsh5864@gmail.com/life_insurance_bel/data/EIOPA_RFR_20251231_Term_Structures.xlsx"
SHEET_NAME = "RFR_spot_no_VA"
VERSION_ID = "RFR_20251231"

# 2행을 헤더로 사용 (header=1 → 0-based index)
df = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME, header=1)

# 컬럼 구조 확인 (디버깅용)
print(df.columns)

# B열 = tenor_year, C열 = Euro
tenor_col = df.columns[1]   # B column
euro_col  = df.columns[2]   # C column

# tenor_year가 숫자인 row만 필터링 (11행 이후 데이터만 남게 됨)
df_filtered = df[df[tenor_col].apply(lambda x: str(x).isdigit())]

df_euro = df_filtered[[tenor_col, euro_col]].copy()
df_euro.columns = ["tenor_year", "zero_rate_annual"]

df_euro["tenor_year"] = df_euro["tenor_year"].astype(int)
df_euro["zero_rate_annual"] = df_euro["zero_rate_annual"].astype(float)
df_euro["tenor_month"] = df_euro["tenor_year"] * 12
df_euro["version_id"] = VERSION_ID


In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
df_spark = spark.createDataFrame(df_euro)

(
    df_spark
    .select("tenor_month", "zero_rate_annual", "version_id")
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.life_insurance_raw.discount_curve")
)
display(df_spark)